In [1]:
import pandas as pd
df = pd.read_csv('books_cleaned.csv')
print("Your books loaded!")
print("Total books:", len(df))
print("\nFirst 5 books:")
print(df[['title', 'authors', 'categories', 'average_rating']].head())


Your books loaded!
Total books: 5197

First 5 books:
                 title                          authors  \
0               Gilead               Marilynne Robinson   
1         Spider's Web  Charles Osborne;Agatha Christie   
2       Rage of angels                   Sidney Sheldon   
3       The Four Loves              Clive Staples Lewis   
4  The Problem of Pain              Clive Staples Lewis   

                      categories  average_rating  
0                        Fiction            3.85  
1  Detective and mystery stories            3.83  
2                        Fiction            3.93  
3                 Christian life            4.15  
4                 Christian life            4.09  


In [2]:
df['search_text'] = (df['title'].fillna('') + ' ' + 
                    df['authors'].fillna('') + ' ' + 
                    df['categories'].fillna('') + ' ' + 
                    df['description'].fillna('')).str.lower()

print("Search text created!")
print("Sample:")
print(df['search_text'].head())


Search text created!
Sample:
0    gilead marilynne robinson fiction a novel that...
1    spider's web charles osborne;agatha christie d...
2    rage of angels sidney sheldon fiction a memora...
3    the four loves clive staples lewis christian l...
4    the problem of pain clive staples lewis christ...
Name: search_text, dtype: object


In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
tfidf_matrix = vectorizer.fit_transform(df['search_text'])
cosine_sim = cosine_similarity(tfidf_matrix)

print("AI Engine built!")
print("Ready to find book recommendations!")


AI Engine built!
Ready to find book recommendations!


In [4]:
content_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)
popularity = df['average_rating'].fillna(0) * df['ratings_count'].fillna(1)
pop_scores = popularity.values / popularity.max()
hybrid_matrix = 0.7 * content_sim + 0.3 * pop_scores[:, np.newaxis]


In [5]:
def test_both(query):
    print(f"\n{'='*60}")
    print(f" Testing: '{query}'")
    print(f"{'='*60}")
    
    content_scores = cosine_similarity(vectorizer.transform([query.lower()]), tfidf_matrix).flatten()
    content_top = np.argsort(content_scores)[-3:][::-1]
    
    print("\n CONTENT-BASED (Pure TF-IDF):")
    for i, idx in enumerate(content_top):
        book = df.iloc[idx]
        print(f"{i+1}. '{book['title']}' by {book['authors']}")
        print(f"   {book['average_rating']:.1f} |  {content_scores[idx]:.0%}")
    
 
    query_vec = vectorizer.transform([query.lower()])
    content_q = cosine_similarity(query_vec, tfidf_matrix).flatten()
    hybrid_q = 0.7 * content_q + 0.3 * pop_scores
    hybrid_top = np.argsort(hybrid_q)[-3:][::-1]
    
    print("\n HYBRID (Content + Popularity):")
    for i, idx in enumerate(hybrid_top):
        book = df.iloc[idx]
        print(f"{i+1}. '{book['title']}' by {book['authors']}")
        print(f"   {book['average_rating']:.1f} |  {hybrid_q[idx]:.0%}")
    
    print(f"{'='*60}\n")


test_both("fantasy")
test_both("Game of Thrones")



 Testing: 'fantasy'

 CONTENT-BASED (Pure TF-IDF):
1. 'Bedlam's Edge' by Mercedes Lackey;Rosemary Edghill
   3.9 |  34%
2. 'The Tombs of Atuan' by Ursula K. Le Guin
   4.1 |  31%
3. 'The Mad Ship' by Robin Hobb
   4.2 |  28%

 HYBRID (Content + Popularity):
1. 'Harry Potter and the Sorcerer's Stone (Book 1)' by Rowling, J.K.
   4.5 |  30%
2. 'Bedlam's Edge' by Mercedes Lackey;Rosemary Edghill
   3.9 |  24%
3. 'The Tombs of Atuan' by Ursula K. Le Guin
   4.1 |  22%


 Testing: 'Game of Thrones'

 CONTENT-BASED (Pure TF-IDF):
1. 'The Glass Bead Game' by Hermann Hesse
   4.1 |  51%
2. 'Monkeewrench' by P. J. Tracy
   4.1 |  44%
3. 'The Player of Games' by Iain Banks
   4.3 |  39%

 HYBRID (Content + Popularity):
1. 'The Glass Bead Game' by Hermann Hesse
   4.1 |  36%
2. 'Monkeewrench' by P. J. Tracy
   4.1 |  31%
3. 'Harry Potter and the Sorcerer's Stone (Book 1)' by Rowling, J.K.
   4.5 |  30%



In [6]:
# Cell 7: PERFECT LAYOUT - Fixed Images + No Overlap!
import gradio as gr
import pandas as pd

def search_with_stats(query):
    """Clean layout - No image overlap!"""
    if not query:
        return gr.update(value="<h2>🔍 Search for books...</h2>")
    
    # Calculate MATCH SCORES (same AI)
    matches = []
    query_words = query.lower().split()
    
    for idx, book in df.iterrows():
        book_text = f"{book.get('title','')} {book.get('authors','')} {book.get('categories','')}".lower()
        word_matches = sum(1 for word in query_words if word in book_text)
        match_pct = min(100, word_matches * 25)
        
        if word_matches > 0:
            matches.append((idx, book, match_pct))
    
    matches.sort(key=lambda x: (x[2], x[1].get('average_rating', 0)), reverse=True)
    top_books = matches[:5]
    
    # PERFECT CLEAN LAYOUT
    html = f"<h2 style='color:#2563eb; margin-bottom:20px'>📚 Top {len(top_books)} for '{query}'</h2>"
    
    for i, (idx, book, match_pct) in enumerate(top_books):
        title = book.get('title', 'No title')
        author = book.get('authors', 'No author')
        rating = book.get('average_rating', 0)
        desc = book.get('description', '')
        thumb = book.get('thumbnail', '')
        
        # FIXED SUMMARY
        summary = "No description" if pd.isna(desc) else str(desc)[:150] + "..."
        
        # FIXED IMAGE + LAYOUT
        img_html = ""
        if thumb and str(thumb).startswith('http'):
            img_html = f"""
            <div style='flex-shrink:0; width:90px; height:135px; margin-right:20px'>
                <img src='{thumb}' style='width:90px; height:135px; object-fit:cover; border-radius:8px; box-shadow:0 2px 8px rgba(0,0,0,0.1)'>
            </div>
            """
        
        html += f"""
        <div style='display:flex; margin-bottom:20px; padding:20px; border:1px solid #e5e7eb; border-radius:12px; background:#f8fafc; min-height:160px'>
            {img_html}
            <div style='flex:1; min-width:0'>
                <h4 style='margin:0 0 8px 0; font-size:18px'>{i+1}. {title}</h4>
                <p style='color:#6b7280; margin:0 0 12px 0; font-size:15px'>{author}</p>
                <p style='color:#059669; font-weight:600; margin:0 0 15px 0; font-size:16px'>
                    ⭐{rating:.1f} | <span style='color:#3b82f6'>📊 {match_pct}% Match</span>
                </p>
                <details style='font-size:14px'>
                    <summary style='cursor:pointer; color:#10b981; font-weight:500; padding:8px 0'>📖 Show Summary</summary>
                    <div style='margin:12px 0 0 0; padding:15px; background:#f0fdf4; border-radius:8px; font-size:14px; line-height:1.5'>
                        {summary}
                    </div>
                </details>
            </div>
        </div>
        """
    
    return gr.update(value=html)

# CLEAN PROFESSIONAL UI
with gr.Blocks(css="""
    .gradio-container {max-width: 1000px !important;}
    """) as demo:
    gr.Markdown("# 📚 AI Book Recommender Pro")
    gr.Markdown("*Search → Match % → Book Summary*")
    
    with gr.Row():
        search_input = gr.Textbox(
            label="🔍 Search Books", 
            placeholder="fantasy dragons, mystery, romance...",
            scale=3
        )
        search_btn = gr.Button("🔍 Find Books", variant="primary", scale=1)
    
    result_output = gr.HTML(value="<h3 style='text-align:center; color:#6b7280'>Start searching for books!</h3>")
    
    search_btn.click(
        fn=search_with_stats,
        inputs=search_input,
        outputs=result_output
    )

print("✅ PERFECT UI ready! No image overlap!")
demo.launch(share=True)


C:\Users\Surface Pro 6\AppData\Local\Temp\ipykernel_12452\2830086728.py:69: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(css="""


✅ PERFECT UI ready! No image overlap!
* Running on local URL:  http://127.0.0.1:7860

Could not create share link. Missing file: C:\Users\Surface Pro 6\.cache\huggingface\gradio\frpc\frpc_windows_amd64_v0.3. 

Please check your internet connection. This can happen if your antivirus software blocks the download of this file. You can install manually by following these steps: 

1. Download this file: https://cdn-media.huggingface.co/frpc-gradio-0.3/frpc_windows_amd64.exe
2. Rename the downloaded file to: frpc_windows_amd64_v0.3
3. Move the file to this location: C:\Users\Surface Pro 6\.cache\huggingface\gradio\frpc
